In [40]:
import numpy as np
import torch
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Modelo de GPU: {torch.cuda.get_device_name(0)}")

corpus = pd.read_csv('C:/Users/inqui/OneDrive/Desktop/Clases/26-2/LLM_PROJECT_1/data/processed_v2/infraestructura_limpio.csv',
                     usecols = ['comentario', 'rango_humano'],encoding = 'utf-8-sig')
print(corpus.head())

GPU disponible: False
                                          comentario  rango_humano
0  cuál es el más cercado para rayar el nombre de...           3.0
1  esos baños deberian estar en el metro, no sabe...           2.0
2  ????????????????????????????????????no pues bu...           3.0
3                                los van a abandonar           2.0
4                           Nada los tiene contentos           4.0


In [41]:
analizador = pipeline(
    "sentiment-analysis",
#    model = 'citizenlab/distilbert-base-multilingual-cased-toxicity' #podria funcionar con un tuneo
#    model ="BAAI/bge-reranker-v2-m3" #no sirve para el objetivo
#    model="distilbert-base-uncased-finetuned-sst-2-english" #el que usa el profe. podria funcionar tuneado
    model = 'nlptown/bert-base-multilingual-uncased-sentiment', #podria funcionar con un tuneo, el mas prometedor por ahora    
#    model = "FacebookAI/roberta-large-mnli", #puede prometer
#    model = 'FacebookAI/xlm-roberta-large', #demasiado crudo
#    model = 'cardiffnlp/twitter-roberta-base-sentiment-latest', #podria ser, pero hay que enseñarle español
#    model = 'Bhumika/roberta-base-finetuned-sst2',
    torch_dtype="auto",           # Detecta automáticamente el tipo óptimo
#    device_map="auto"             # Coloca el modelo en GPU si está disponible
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [42]:
etiqueta = []
confianza = []

for texto in list(corpus['comentario']):
    try:
        aux = analizador(texto)[0]
        etiqueta.append(aux['label'])
        confianza.append(aux['score'])
    except:
        print(texto, type(texto))

Vamos ahora con el calculo de las metricas del baseline

In [43]:
resultados = pd.DataFrame({'Comentario':list(corpus['comentario']),'Puntuacion Estimada': [float(etiq[0]) for etiq in etiqueta],
                           'Puntuacion Real':corpus['rango_humano'],'Confianza':confianza})

print(resultados)
#resultados.to_csv('C:/Users/inqui/OneDrive/Desktop/borrar.csv')
print('--------------------------------------------------------------------')
#print(resultados.dropna())


#aqui se hace la comparacion promedio de atinados y predichos, matriz de confusion, etc...


                                            Comentario  Puntuacion Estimada  \
0    cuál es el más cercado para rayar el nombre de...                  5.0   
1    esos baños deberian estar en el metro, no sabe...                  1.0   
2    ????????????????????????????????????no pues bu...                  1.0   
3                                  los van a abandonar                  1.0   
4                             Nada los tiene contentos                  1.0   
..                                                 ...                  ...   
896                                         ??????????                  1.0   
897  Hay al ratito van a decir que costo mil de mil...                  1.0   
898  Pov comienzan a cerrar estaciones estoy segura...                  1.0   
899                      esos serán moteles no se agan                  1.0   
900                                         comentario                  3.0   

     Puntuacion Real  Confianza  
0                

In [50]:
muestra = resultados[np.isnan(corpus['rango_humano']) == False]
# print(muestra)
semilla = 61298

print(classification_report(muestra['Puntuacion Estimada'], muestra['Puntuacion Real'],
                             target_names = ['negativa','parcialmente negativo',
                             'neutral','parcialmente negativo', 'positiva']))

# Matriz de confusión
orden = ['negativa','parcialmente negativo',
        'neutral','parcialmente negativo', 'positiva']
cm = confusion_matrix(muestra['Puntuacion Estimada'], muestra['Puntuacion Real'], labels=orden)


fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(orden, fontsize=11)
ax.set_yticks(range(3)); ax.set_yticklabels(orden, fontsize=11)
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_title('Matriz de Confusión — Baseline\n(dataset de ejemplo, 12 reseñas)',
             fontsize=11, fontweight='bold')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.show()


                       precision    recall  f1-score   support

             negativa       0.82      0.19      0.31       139
parcialmente negativo       0.14      0.58      0.23        19
              neutral       0.07      0.38      0.11        16
parcialmente negativo       0.12      0.12      0.12        17
             positiva       0.00      0.00      0.00        29

             accuracy                           0.21       220
            macro avg       0.23      0.25      0.15       220
         weighted avg       0.54      0.21      0.24       220



ValueError: At least one label specified must be in y_true

In [ ]:
#k-cross validation para fine tuning
def kfold_cross_validation(df, k=5, shuffle=True, random_state=None, stratify=False):
    """
    Genera particiones para k-fold cross validation.
    
    Parámetros:
    -----------
    df : DataFrame
        DataFrame a particionar
    k : int
        Número de folds (particiones)
    shuffle : bool
        Si mezclar los datos antes de particionar
    random_state : int
        Semilla para reproducibilidad
    stratify : bool o Series
        Si es True, usa la primera columna como estratificación
        Si es Series, usa esa columna para estratificar
    
    Retorna:
    --------
    list : Lista de tuplas (train_indices, test_indices)
    """
    
    if stratify:
        # Obtener etiquetas para estratificación
        if isinstance(stratify, bool):
            y = df.iloc[:, 1]  # Usa primera columna
        else:
            y = df[stratify]
        kfold = StratifiedKFold(n_splits=k, shuffle=shuffle, random_state=random_state)
    else:
        kfold = KFold(n_splits=k, shuffle=shuffle, random_state=random_state)
    
    # Generar folds
    folds = list(kfold.split(df, y if stratify else None))
    
    return folds

# K-Fold simple

folds = kfold_cross_validation(muestra, k=5, shuffle=True, random_state=42)
#print(folds)


etiqueta_train = []
confianza_train = []


for i, (train_idx, test_idx) in enumerate(folds):
    #print(f"Fold {i+1}: Train={len(train_idx)}, Test={len(test_idx)}")
    #print(train_idx)
    #print(test_idx)
    # Obtenemos los DataFrames
    train_df = muestra.iloc[train_idx]
    test_df = muestra.iloc[test_idx]
    
    

    pass

[  0   1   2   3   4   5   6   7   8  10  11  12  13  14  17  19  20  21
  22  23  24  26  27  28  29  31  32  33  34  35  36  37  38  39  40  41
  42  43  44  46  47  48  49  50  51  52  53  54  56  57  58  59  60  61
  62  63  64  65  68  69  70  71  72  74  76  77  78  79  80  81  83  84
  85  87  88  89  90  91  92  94  98  99 102 103 104 105 106 107 110 112
 113 114 116 117 118 119 121 122 123 124 125 126 128 129 130 131 133 134
 135 136 137 139 140 143 144 145 146 147 149 150 151 152 153 155 156 157
 158 159 160 161 162 163 164 165 166 167 168 169 170 171 173 174 176 177
 178 179 181 182 183 184 185 186 187 188 191 192 194 195 198 199 200 202
 203 204 205 206 207 208 211 212 213 214 215 216 217 219]
[  9  15  16  18  25  30  45  55  66  67  73  75  82  86  93  95  96  97
 100 101 108 109 111 115 120 127 132 138 141 142 148 154 172 175 180 189
 190 193 196 197 201 209 210 218]
[  0   1   2   3   4   6   7   8   9  10  11  13  14  15  16  17  18  20
  21  22  23  25  27  30  32  33